# Identificação de Espécies de Pássaros com Visão Computacional Clássica

**Disciplina:** INE410121 / TRV410001 — Visão Computacional · UFSC
**Dataset:** Backyard Feeder Birds (NABirds Subset)
**Objetivo:** Construir uma pipeline clássica de VC para classificação de espécies de pássaros, sem uso de Deep Learning.

## Pipeline
1. Configuração do ambiente e carregamento do dataset
2. Exploração do dataset
3. Pré-processamento das imagens
4. Segmentação clássica com múltiplos candidatos
5. Extração de features clássicas sobre a região segmentada
6. Treinamento de classificador supervisionado
7. Avaliação

# 1. Configuração do Ambiente
## 1.1. Setup: ambiente e dataset

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.setup import setup

DATA_DIR = setup()
print("Dataset root:", DATA_DIR)


## 1.2. Importações e configurações globais

Carregamos todas as bibliotecas necessárias e definimos o diretório de saída desta execução.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

from pathlib import Path
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC

from src.utils import create_run_dir, save_fig, save_metrics

%matplotlib inline
matplotlib.rcParams["figure.figsize"] = (14, 6)

RUN_DIR = create_run_dir()


# 2. Exploração do Dataset
## 2.1. Descoberta automática da estrutura

O dataset não tem um manifesto. Esta célula percorre as subpastas e assume que cada subpasta é uma classe, identificando-a pelo nome.

In [ ]:
DATASET = Path(DATA_DIR)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def find_class_folders(root: Path):
    class_dirs = []
    for subdir in root.rglob("*"):
        if subdir.is_dir():
            try:
                files = [p for p in subdir.iterdir() if p.suffix.lower() in IMAGE_EXTS]
                if files:
                    class_dirs.append(subdir)
            except Exception:
                pass
    return sorted(set(class_dirs))

class_dirs = find_class_folders(DATASET)
print(f"Found {len(class_dirs)} candidate class folders")
for d in class_dirs[:20]:
    print(d)


### Montagem do DataFrame com todas as imagens encontradas

In [ ]:
rows = []
for class_dir in class_dirs:
    class_name = class_dir.name
    for img_path in class_dir.iterdir():
        if img_path.suffix.lower() in IMAGE_EXTS:
            rows.append({
                "class_name": class_name,
                "image_path": str(img_path)
            })

df = pd.DataFrame(rows)
print(df.head())
print()
print("Total images:", len(df))
print("Total classes:", df["class_name"].nunique())


## 2.2. Distribuição das classes

Visualizamos quantas imagens existem por espécie para entender o balanceamento do dataset.

In [ ]:
class_counts = df["class_name"].value_counts()

plt.figure(figsize=(14, 5))
class_counts.plot(kind="bar")
plt.title("Número de imagens por classe")
plt.ylabel("Quantidade")
plt.xlabel("Classe")
plt.xticks(rotation=90)
plt.tight_layout()
save_fig("01_class_distribution", RUN_DIR)
plt.show()


## 2.3. Amostras do dataset

Exibimos até duas imagens por classe para inspecionar a variabilidade visual.

In [ ]:
def load_rgb(path: str):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

sample_df = df.groupby("class_name").head(2).reset_index(drop=True)
n = min(len(sample_df), 12)
sample_df = sample_df.iloc[:n]

rows_plot = int(np.ceil(n / 2))
fig, axes = plt.subplots(rows_plot, 2, figsize=(10, 5 * rows_plot))
axes = np.array(axes).reshape(-1)

for ax, (_, row) in zip(axes, sample_df.iterrows()):
    img = load_rgb(row["image_path"])
    ax.imshow(img)
    ax.set_title(row["class_name"])
    ax.axis("off")

for ax in axes[len(sample_df):]:
    ax.axis("off")

plt.tight_layout()
save_fig("02_samples", RUN_DIR)
plt.show()


## 2.4. Galeria completa — uma amostra por espécie

In [ ]:
top_25 = df["class_name"].value_counts().head(25).index.tolist()
gallery_df = (
    df[df["class_name"].isin(top_25)]
    .groupby("class_name")
    .first()
    .reset_index()
)

N_COLS = 5
n_rows = int(np.ceil(len(gallery_df) / N_COLS))
fig, axes = plt.subplots(n_rows, N_COLS, figsize=(N_COLS * 3, n_rows * 3.2))
axes = np.array(axes).reshape(-1)

for ax, (_, row) in zip(axes, gallery_df.iterrows()):
    img = load_rgb(row["image_path"])
    ax.imshow(img)
    ax.set_title(row["class_name"].replace("_", "\n"), fontsize=7)
    ax.axis("off")

for ax in axes[len(gallery_df):]:
    ax.axis("off")

plt.suptitle("Galeria de classes — uma imagem por espécie", fontsize=13)
plt.tight_layout()
save_fig("02b_class_gallery", RUN_DIR)
plt.show()

## 2.5. Variabilidade intra-classe

Mostramos 6 fotos da **mesma espécie** para evidenciar o desafio: pose, escala, fundo e iluminação variam drasticamente. Este é o argumento central de por que o problema é difícil para métodos clássicos.

In [ ]:
import os

# 4 espécies com boa quantidade de imagens, escolhidas pela diversidade visual
species_to_show = top_25[:4]
N_PER_SPECIES = 6

fig, axes = plt.subplots(len(species_to_show), N_PER_SPECIES,
                         figsize=(N_PER_SPECIES * 2.8, len(species_to_show) * 2.8))

for row_idx, species in enumerate(species_to_show):
    imgs = df[df["class_name"] == species]["image_path"].tolist()

    # Amostras distribuídas ao longo de todo o conjunto para máxima variedade
    indices = np.linspace(0, len(imgs) - 1, N_PER_SPECIES, dtype=int)
    samples = [imgs[i] for i in indices]

    for col_idx, img_path in enumerate(samples):
        ax = axes[row_idx, col_idx]
        ax.imshow(load_rgb(img_path))
        ax.axis("off")

    # Rótulo da espécie na primeira coluna
    axes[row_idx, 0].set_title(species.replace("_", " "), fontsize=8,
                               loc="left", pad=4, fontweight="bold")

plt.suptitle("Variabilidade intra-classe — mesma espécie, condições muito diferentes",
             fontsize=12, y=1.01)
plt.tight_layout()

# Salva em caminho fixo (sem timestamp) para uso na apresentação
OUT_PATH = os.path.join("..", "outputs", "figures", "dificuldade.png")
fig.savefig(OUT_PATH, dpi=120, bbox_inches="tight")
plt.show()
print(f"Salvo em {OUT_PATH}")

# 3. Pré-processamento
## 3.1. Pipeline de pré-processamento

Cada imagem é redimensionada para `IMG_SIZE = (160, 160)` e convertida para três representações:
- **RGB** — para exibição e para o GrabCut;
- **HSV** — para features de cor e o candidato de saturação;
- **Escala de cinza + blur gaussiano** — para os candidatos baseados em limiarização.

In [ ]:
IMG_SIZE = (160, 160)

def preprocess_image(path: str):
    img_bgr = cv2.imread(path)
    img_bgr = cv2.resize(img_bgr, IMG_SIZE)

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    img_blur = cv2.GaussianBlur(img_gray, (5, 5), 0)

    return img_rgb, img_hsv, img_gray, img_blur


# 4. Segmentação Clássica com Múltiplos Candidatos

A estratégia adotada é gerar **seis máscaras candidatas** com métodos distintos e, em seguida, escolher automaticamente a melhor pelo critério de score definido em `mask_score`.

Essa abordagem evita que um único método com limitações específicas (ex: Otsu falha em pássaros escuros) prejudique o resultado geral.

## 4.1. Funções auxiliares: limpeza de máscara

- `largest_component`: mantém apenas a maior componente conexa, eliminando ruído fragmentado.
- `clean_mask`: aplica abertura (remove ruído miúdo) e fechamento (preenche buracos internos), depois chama `largest_component`.

In [ ]:
def largest_component(mask):
    """Mantém apenas a maior componente conexa da máscara binária."""
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels <= 1:
        return mask
    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_idx = 1 + np.argmax(areas)
    out = np.zeros_like(mask)
    out[labels == largest_idx] = 255
    return out


def clean_mask(mask, k_open=3, k_close=9):
    """
    Pós-processamento morfológico comum a todos os candidatos.
    k_open  — abertura: remove ruídos pequenos isolados.
    k_close — fechamento: preenche buracos internos da região segmentada.
    """
    kernel_open  = np.ones((k_open,  k_open),  np.uint8)
    kernel_close = np.ones((k_close, k_close), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel_open)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)
    return largest_component(mask)

## 4.2. Candidatos de limiarização global: Otsu

O método de Otsu encontra automaticamente o limiar que maximiza a variância entre dois grupos de pixels.
Geramos duas versões — direta e invertida — para cobrir casos em que o pássaro é claro *ou* escuro em relação ao fundo.

In [ ]:
def segment_otsu(img_blur):
    """Limiarização global de Otsu (pássaro como região clara)."""
    _, mask = cv2.threshold(img_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return clean_mask(mask)


def segment_otsu_inverse(img_blur):
    """Limiarização global de Otsu invertida (pássaro como região escura)."""
    _, mask = cv2.threshold(img_blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return clean_mask(mask)

## 4.3. Candidato: limiar adaptativo gaussiano

O limiar adaptativo calcula um valor de corte local para cada pixel com base na média ponderada (gaussiana) da sua vizinhança. Tende a funcionar melhor quando a iluminação é não uniforme.

In [ ]:
def segment_adaptive(img_blur):
    """Limiarização adaptativa com ponderação gaussiana (blockSize=21, C=4)."""
    mask = cv2.adaptiveThreshold(
        img_blur, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 21, 4
    )
    return clean_mask(mask)

## 4.4. Candidato: detecção de bordas (Canny + morfologia)

Detecta bordas fortes com Canny, dilata-as para criar uma região contínua e aplica fechamento morfológico para tentar fechar o contorno do objeto.

In [ ]:
def segment_edges(img_blur):
    """Máscara baseada em bordas Canny, dilatadas e fechadas morfologicamente."""
    edges = cv2.Canny(img_blur, 60, 140)
    kernel = np.ones((3, 3), np.uint8)
    # Dilata as bordas para torná-las mais espessas e conectadas
    mask = cv2.dilate(edges, kernel, iterations=2)
    # Fecha o contorno para criar uma região preenchida
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    return clean_mask(mask)

## 4.5. Candidato: saturação HSV

Separa o objeto do fundo por cor, combinando dois critérios no espaço HSV:
1. **Pixels coloridos** (S ≥ 40) — captura aves com plumagem vistosa;
2. **Pixels escuros** (V < 110, S < 60) — captura aves de cor escura como o Peru Selvagem.

In [ ]:
def segment_hsv_saturation(img_hsv):
    """
    Combina dois intervalos HSV:
    - colorful: pixels com alta saturação (aves coloridas);
    - dark_fg:  pixels escuros (aves negras/acinzentadas).
    """
    colorful = cv2.inRange(img_hsv, (0,  40,  20), (179, 255, 255))
    dark_fg  = cv2.inRange(img_hsv, (0,   0,  10), (179,  60, 110))
    mask = cv2.bitwise_or(colorful, dark_fg)
    return clean_mask(mask)

## 4.6. Candidato: GrabCut

O GrabCut é um algoritmo iterativo que estima modelos de cor do fundo e do objeto dentro de um retângulo de inicialização, refinando a segmentação por grafos.

Usamos margem de 15% em cada borda e 7 iterações para equilibrar foco no sujeito e qualidade de refinamento.

In [ ]:
def segment_grabcut_rect(img_rgb):
    """
    GrabCut com retângulo central (margem 15%, 7 iterações).
    Retorna máscara binária com os pixels classificados como foreground.
    """
    img = img_rgb.copy()
    mask = np.zeros(img.shape[:2], np.uint8)
    h, w = img.shape[:2]
    # Retângulo que delimita a região de interesse (exclui 15% das bordas)
    rect = (int(w * 0.15), int(h * 0.15), int(w * 0.70), int(h * 0.70))
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    try:
        cv2.grabCut(img, mask, rect, bgd_model, fgd_model, 7, cv2.GC_INIT_WITH_RECT)
        out = np.where(
            (mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0
        ).astype("uint8")
        return clean_mask(out)
    except Exception:
        return np.zeros(img.shape[:2], dtype=np.uint8)

## 4.7. Critério de seleção: score da máscara

O score combina quatro critérios para identificar automaticamente a melhor máscara candidata:

| Critério | Pontuação |
|---|---|
| Razão de área dentro de `[0.03, 0.65]` | +2.0 (fora: −2.0) |
| Fração da maior componente conexa | +[0, 1] |
| Proximidade do centróide ao centro da imagem | +[0, 0.5] |
| Compacidade `4πA/P²` (penaliza máscaras recortadas) | +[0, 1] |

In [ ]:
def mask_score(mask):
    """Calcula um score de qualidade para uma máscara candidata."""
    area = np.count_nonzero(mask)
    h, w = mask.shape
    area_ratio = area / (h * w + 1e-8)

    if area == 0:
        return -1e9

    score = 0.0

    # Bônus/penalidade pela razão de área
    if 0.03 <= area_ratio <= 0.65:
        score += 2.0
    else:
        score -= 2.0

    # Bônus pela compactação em uma única componente
    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels > 1:
        largest_area = stats[1:, cv2.CC_STAT_AREA].max()
        score += largest_area / (area + 1e-8)

    # Bônus de proximidade ao centro (pássaros tendem a ser o sujeito central)
    ys, xs = np.where(mask > 0)
    cy, cx = ys.mean() / h, xs.mean() / w
    dist = ((cy - 0.5) ** 2 + (cx - 0.5) ** 2) ** 0.5
    score += max(0.0, 0.5 - dist)

    # Bônus de compacidade: 4π·área/perímetro² (1 = círculo perfeito)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        a_c = cv2.contourArea(c)
        p_c = cv2.arcLength(c, True)
        if p_c > 0:
            score += 4 * np.pi * a_c / (p_c ** 2)

    return score

## 4.8. Pipeline completa: `segment_bird`

Agrega as funções anteriores em uma única chamada: dado o caminho da imagem, retorna um dicionário com todos os candidatos e a melhor máscara selecionada.

In [ ]:
def build_segmentation_candidates(img_rgb, img_hsv, img_gray, img_blur):
    """Gera os seis candidatos de segmentação para uma imagem já pré-processada."""
    return {
        "otsu":     segment_otsu(img_blur),
        "otsu_inv": segment_otsu_inverse(img_blur),
        "adaptive": segment_adaptive(img_blur),
        "edges":    segment_edges(img_blur),
        "hsv_sat":  segment_hsv_saturation(img_hsv),
        "grabcut":  segment_grabcut_rect(img_rgb),
    }


def choose_best_mask(candidates):
    """Retorna o nome, a máscara e o score do candidato com maior pontuação."""
    best_name, best_mask, best_score = None, None, -1e18
    for name, mask in candidates.items():
        score = mask_score(mask)
        if score > best_score:
            best_score, best_name, best_mask = score, name, mask
    return best_name, best_mask, best_score


def segment_bird(path: str) -> dict:
    """Pipeline completa: carrega → pré-processa → gera candidatos → seleciona melhor máscara."""
    img_rgb, img_hsv, img_gray, img_blur = preprocess_image(path)
    candidates = build_segmentation_candidates(img_rgb, img_hsv, img_gray, img_blur)
    best_name, best_mask, best_score = choose_best_mask(candidates)

    # Aplica a máscara à imagem original para produzir o segmento final
    segmented = img_rgb.copy()
    segmented[best_mask == 0] = 0

    return {
        "rgb": img_rgb, "hsv": img_hsv, "gray": img_gray, "blur": img_blur,
        "candidates": candidates,
        "best_name": best_name, "best_mask": best_mask, "best_score": best_score,
        "segmented": segmented,
    }

## 4.9. Visualização dos candidatos para uma imagem

A função abaixo exibe uma galeria comparando todos os seis candidatos ao mesmo tempo, facilitando a inspeção visual da seleção feita pelo score.

In [ ]:
def show_segmentation_gallery(path: str):
    result = segment_bird(path)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    axes[0].imshow(result["rgb"])
    axes[0].set_title("RGB")
    axes[0].axis("off")

    axes[1].imshow(result["gray"], cmap="gray")
    axes[1].set_title("Gray")
    axes[1].axis("off")

    names = list(result["candidates"].keys())
    for i, name in enumerate(names[:4], start=2):
        axes[i].imshow(result["candidates"][name], cmap="gray")
        axes[i].set_title(name)
        axes[i].axis("off")

    for ax_idx, name in zip([6, 7], names[4:6]):
        axes[ax_idx].imshow(result["candidates"][name], cmap="gray")
        axes[ax_idx].set_title(name)
        axes[ax_idx].axis("off")

    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(result["rgb"])
    ax[0].set_title("Original")
    ax[0].axis("off")

    ax[1].imshow(result["best_mask"], cmap="gray")
    ax[1].set_title(f"Best mask: {result['best_name']}")
    ax[1].axis("off")

    ax[2].imshow(result["segmented"])
    ax[2].set_title("Segmented bird")
    ax[2].axis("off")

    plt.tight_layout()
    plt.show()

    return result


### Exemplo interativo: inspecionar um pássaro aleatório

In [ ]:
row = df.sample(1, random_state=42).iloc[0]
print(row["class_name"])
seg_result = show_segmentation_gallery(row["image_path"])


## 4.10. Galeria de exemplos segmentados

Seis imagens aleatórias, mostrando original, máscara escolhida e o resultado final segmentado.

In [ ]:
sample_paths = df.sample(6, random_state=123).reset_index(drop=True)

fig, axes = plt.subplots(6, 3, figsize=(12, 24))

for i, (_, row) in enumerate(sample_paths.iterrows()):
    res = segment_bird(row["image_path"])

    axes[i, 0].imshow(res["rgb"])
    axes[i, 0].set_title(f"Original\n{row['class_name']}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(res["best_mask"], cmap="gray")
    axes[i, 1].set_title(f"Mask ({res['best_name']})")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(res["segmented"])
    axes[i, 2].set_title("Segmented")
    axes[i, 2].axis("off")

plt.tight_layout()
save_fig("03_segmentation_examples", RUN_DIR)
plt.show()


## 4.11. Comparação lado a lado de todos os candidatos

Cada linha exibe uma imagem com seus seis candidatos e o segmento final. A estrela (★) indica o método selecionado pelo score.

In [ ]:
_METHOD_NAMES = ["otsu", "otsu_inv", "adaptive", "edges", "hsv_sat", "grabcut"]

# Seed 15 escolhe espécies variadas (evita o Peru Selvagem que é caso difícil)
sample_rows = df.sample(4, random_state=15).reset_index(drop=True)

# 4 imagens × 8 colunas (original, 6 candidatos, segmentado)
fig, axes = plt.subplots(4, 8, figsize=(24, 12))

for row_idx, (_, row) in enumerate(sample_rows.iterrows()):
    res = segment_bird(row["image_path"])

    axes[row_idx, 0].imshow(res["rgb"])
    axes[row_idx, 0].set_title(f"Original\n{row['class_name']}", fontsize=6)
    axes[row_idx, 0].axis("off")

    for col_idx, name in enumerate(_METHOD_NAMES, start=1):
        mask = res["candidates"][name]
        axes[row_idx, col_idx].imshow(mask, cmap="gray")
        label = f"★ {name}" if name == res["best_name"] else name
        axes[row_idx, col_idx].set_title(label, fontsize=6)
        axes[row_idx, col_idx].axis("off")

    axes[row_idx, 7].imshow(res["segmented"])
    axes[row_idx, 7].set_title(f"Segmentado\n({res['best_name']})", fontsize=6)
    axes[row_idx, 7].axis("off")

plt.suptitle("Candidatos de segmentação — ★ = escolhido pelo score", fontsize=12)
plt.tight_layout()
save_fig("03b_candidates_comparison", RUN_DIR)
plt.show()

## 4.12. Qual método é escolhido com maior frequência?

Rodamos a pipeline em 100 imagens aleatórias e contamos quantas vezes cada candidato vence o critério de score. Também identificamos casos com segmentação difícil (máscara fora do intervalo de área esperado).

**Discussão:** Por que o *adaptive* nunca é escolhido? Qual característica do score mais penaliza esse método?

In [ ]:
_METHOD_NAMES = ["otsu", "otsu_inv", "adaptive", "edges", "hsv_sat", "grabcut"]

# --- Frequência de escolha por método ---
method_counts = {m: 0 for m in _METHOD_NAMES}
difficult = []

for _, row in tqdm(df.sample(100, random_state=99).iterrows(), total=100, desc="Analisando"):
    try:
        res = segment_bird(row["image_path"])
        method_counts[res["best_name"]] += 1
        area_ratio = np.count_nonzero(res["best_mask"]) / res["best_mask"].size
        if area_ratio < 0.03 or area_ratio > 0.70:
            difficult.append({"row": row, "res": res, "ratio": area_ratio})
    except Exception:
        pass

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(method_counts.keys(), method_counts.values(), color="steelblue", edgecolor="white")
ax.set_title("Método de segmentação escolhido com maior frequência (n=100)")
ax.set_ylabel("Contagem")
ax.set_xlabel("Método")
for bar in ax.patches:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        str(int(bar.get_height())),
        ha="center", va="bottom", fontsize=9
    )
plt.tight_layout()
save_fig("03c_method_frequency", RUN_DIR)
plt.show()

# --- Casos com segmentação difícil ---
n_show = min(6, len(difficult))
print(f"\nCasos com máscara fora do intervalo ideal: {len(difficult)}/100")

if n_show > 0:
    fig, axes = plt.subplots(n_show, 3, figsize=(10, n_show * 3))
    if n_show == 1:
        axes = axes.reshape(1, -1)
    for i, entry in enumerate(difficult[:n_show]):
        axes[i, 0].imshow(entry["res"]["rgb"])
        axes[i, 0].set_title(f"Original\n{entry['row']['class_name']}", fontsize=8)
        axes[i, 0].axis("off")

        axes[i, 1].imshow(entry["res"]["best_mask"], cmap="gray")
        axes[i, 1].set_title(
            f"Máscara — {entry['res']['best_name']}\nratio={entry['ratio']:.2f}", fontsize=8
        )
        axes[i, 1].axis("off")

        axes[i, 2].imshow(entry["res"]["segmented"])
        axes[i, 2].set_title("Segmentado", fontsize=8)
        axes[i, 2].axis("off")

    plt.suptitle("Casos com segmentação difícil (ratio < 0.03 ou > 0.70)", fontsize=11)
    plt.tight_layout()
    save_fig("03d_difficult_cases", RUN_DIR)
    plt.show()

# 5. Extração de Features Clássicas

Após segmentar o pássaro, descrevemos a região identificada por quatro grupos de atributos visuais. Cada grupo captura um aspecto diferente da aparência do objeto.

## 5.1. Features de cor — histograma HSV

O espaço HSV separa matiz (H), saturação (S) e valor (V), tornando os histogramas mais robustos a variações de iluminação do que RGB.
Calculamos 16 bins por canal → vetor de 48 dimensões.

In [ ]:
def color_features_masked(img_hsv, mask):
    feats = []

    for ch, bins, rng in [(0, 16, [0, 180]), (1, 16, [0, 256]), (2, 16, [0, 256])]:
        hist = cv2.calcHist([img_hsv], [ch], mask, [bins], rng).flatten()
        hist = hist.astype(np.float32)
        hist /= (hist.sum() + 1e-8)
        feats.append(hist)

    return np.concatenate(feats).astype(np.float32)

## 5.2. Features de textura — LBP manual

O **Local Binary Pattern (LBP)** codifica a estrutura local de textura comparando cada pixel com seus 8 vizinhos.
Implementamos o LBP "from scratch" para fins didáticos, depois calculamos seu histograma (32 bins) sobre a região da máscara.

In [ ]:
def lbp_image(gray):
    """Calcula o mapa LBP (8 vizinhos) para toda a imagem em escala de cinza."""
    h, w = gray.shape
    lbp = np.zeros((h - 2, w - 2), dtype=np.uint8)
    center = gray[1:-1, 1:-1]
    offsets = [(-1,-1),(-1,0),(-1,1),(0,1),(1,1),(1,0),(1,-1),(0,-1)]
    for i, (dy, dx) in enumerate(offsets):
        neighbor = gray[1+dy : h-1+dy, 1+dx : w-1+dx]
        lbp |= ((neighbor >= center) << i).astype(np.uint8)
    return lbp


def texture_features_masked(img_gray, mask):
    """Histograma de LBP (32 bins) calculado apenas sobre os pixels da máscara."""
    lbp = lbp_image(img_gray)
    mask_inner = mask[1:-1, 1:-1]   # LBP perde 1 pixel em cada borda
    vals = lbp[mask_inner > 0]
    if len(vals) == 0:
        return np.zeros(32, dtype=np.float32)
    hist, _ = np.histogram(vals, bins=32, range=(0, 256))
    hist = hist.astype(np.float32)
    hist /= (hist.sum() + 1e-8)
    return hist

## 5.3. Features de forma

Descrevemos a geometria da máscara com atributos clássicos de visão computacional:
área, perímetro, aspect ratio, extensão (extent), solidez (convex hull), diâmetro equivalente e os 7 **Momentos de Hu** (invariantes à translação, escala e rotação).

In [ ]:
def shape_features_from_mask(mask):
    """Extrai descritores geométricos do contorno externo da máscara (12 valores)."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return np.zeros(12, dtype=np.float32)

    c = max(contours, key=cv2.contourArea)
    area      = cv2.contourArea(c)
    perimeter = cv2.arcLength(c, True)

    x, y, w, h = cv2.boundingRect(c)
    aspect_ratio = w / (h + 1e-8)
    extent       = area / (w * h + 1e-8)           # fração preenchida do bounding box

    hull      = cv2.convexHull(c)
    hull_area = cv2.contourArea(hull)
    solidity  = area / (hull_area + 1e-8)           # razão área/casco convexo

    eq_diam = np.sqrt(4 * area / np.pi) if area > 0 else 0.0

    # Momentos de Hu: invariantes à translação, escala e rotação
    moments = cv2.moments(c)
    hu = cv2.HuMoments(moments).flatten()
    hu = np.sign(hu) * np.log1p(np.abs(hu))        # log para comprimir a escala

    return np.concatenate([
        np.array([area, perimeter, aspect_ratio, extent, solidity, eq_diam],
                 dtype=np.float32),
        hu.astype(np.float32)
    ])

## 5.4. Features de bordas e gradiente

Calculamos a **densidade de bordas** dentro da máscara (razão entre pixels de borda e pixels de objeto) e um histograma da magnitude do gradiente de Sobel (8 bins + 1 = 9 valores).

In [ ]:
def edge_features_masked(img_gray, mask):
    """Density de bordas Canny + histograma de gradiente Sobel (9 valores)."""
    edges = cv2.Canny(img_gray, 60, 140)
    edge_ratio = (
        np.count_nonzero((edges > 0) & (mask > 0)) / (np.count_nonzero(mask) + 1e-8)
    )

    gx  = cv2.Sobel(img_gray, cv2.CV_32F, 1, 0, ksize=3)
    gy  = cv2.Sobel(img_gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = cv2.magnitude(gx, gy)

    vals = mag[mask > 0]
    if len(vals) == 0:
        return np.zeros(9, dtype=np.float32)

    hist, _ = np.histogram(vals, bins=8, range=(0, np.max(vals) + 1e-8))
    hist = hist.astype(np.float32) / (hist.sum() + 1e-8)
    return np.concatenate([hist, np.array([edge_ratio], dtype=np.float32)])

## 5.5. Vetor completo de features

Combina os quatro grupos em um único vetor de **101 dimensões** por imagem:
48 (cor) + 32 (textura) + 12 (forma) + 9 (bordas).

In [ ]:
def extract_features(path: str):
    """Extrai o vetor de 101 features clássicas de uma imagem segmentada."""
    result = segment_bird(path)

    img_hsv  = result["hsv"]
    img_gray = result["gray"]
    mask     = result["best_mask"]

    f_color   = color_features_masked(img_hsv,  mask)   # 48
    f_texture = texture_features_masked(img_gray, mask)  # 32
    f_shape   = shape_features_from_mask(mask)           # 12
    f_edge    = edge_features_masked(img_gray, mask)     # 9

    return np.concatenate([f_color, f_texture, f_shape, f_edge]).astype(np.float32)

# 6. Classificação com SVM
## 6.1. Seleção da POC

Trabalhamos com as 25 classes mais frequentes, limitando a 60 imagens por classe, para manter o experimento tratável em CPU.

In [ ]:
TOP_N_CLASSES = 25
MAX_PER_CLASS = 60

top_classes = df["class_name"].value_counts().head(TOP_N_CLASSES).index

df_small = (
    df[df["class_name"].isin(top_classes)]
      .groupby("class_name", group_keys=False)
      .head(MAX_PER_CLASS)
      .reset_index(drop=True)
)

print("Images in POC:", len(df_small))
print("Classes in POC:", df_small["class_name"].nunique())


## 6.2. Montagem do dataset de features

Iteramos sobre todas as imagens selecionadas, extraímos o vetor de features e montamos as matrizes `X` e `y`.

In [ ]:
X = []
y = []

for _, row in tqdm(df_small.iterrows(), total=len(df_small)):
    try:
        feat = extract_features(row["image_path"])
        X.append(feat)
        y.append(row["class_name"])
    except Exception as e:
        print("Skipping:", row["image_path"], e)

X = np.array(X, dtype=np.float32)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)


## 6.3. Treinamento e avaliação — SVM RBF

Dividimos em treino/teste (80/20), normalizamos com `StandardScaler` e treinamos um **SVM com kernel RBF**.
O relatório de classificação mostra precisão, recall e F1 por espécie.

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=5, gamma="scale"))
])

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))


## 6.4. Matriz de confusão

Cada linha representa a classe real, cada coluna a classe predita. A diagonal principal indica acertos.

**Discussão:** Quais espécies são mais confundidas entre si? Isso se deve à semelhança visual ou a falhas na segmentação?

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
save_fig("04_confusion_matrix", RUN_DIR)
plt.show()


# 7. Salvando Métricas

Registramos os parâmetros e métricas desta execução em JSON para reprodutibilidade.

In [ ]:
metrics = {
    "dataset_slug": "jakemccaig/backyard-feeder-birds-nabirds-subset",
    "num_images": int(len(X)),
    "num_classes": int(len(le.classes_)),
    "feature_dim": int(X.shape[1]),
    "classifier": "SVM",
    "accuracy": float(acc)
}

save_metrics(metrics, RUN_DIR)


# 8. Próximos Passos

Possíveis extensões desta POC:
- Comparar SVM × kNN × Floresta Aleatória usando os mesmos descritores
- Avaliar o impacto de cada grupo de features isoladamente (ablation study)
- Testar outros candidatos de segmentação (watershed, mean-shift)
- Incorporar métricas de similaridade no espaço de cor (Lab, distância de Mahalanobis) para melhorar o score da máscara
- Explorar o espaço HSV de forma sistemática para identificar cromaticidades específicas de cada espécie